#Trabalho de Conclusão de Curso
#Especialização em Ciência de Dados
Prof. Eduardo Kugler Viegas<BR>
Alunos: Humberto Pradera e Leonardo Rocha

Experimentos para subsidiar a escrita do artigo<BR>

Utilizamos a versão NetFlow v3 Datasets

Visualização dos gráficos e experimento crossdataset

# Preâmbulo

# Imports, variáveis e funções gerais

In [1]:
#Bibliotecas
import numpy as np
import pandas as pd
import copy #copiar o modelo em memória


import matplotlib.pyplot as plt


from google.colab import drive
import math
from time import time as time2
import datetime
#import time



In [2]:
# Controle de alguns experimentos
DATA_RESULTADOS = "2025-12-11 200"
#DATA_RESULTADOS = "inspecao"

LOG_DIR = f'/content/gdrive/MyDrive/Colab Notebooks/Especialização/TCC/Resultados/{DATA_RESULTADOS}/'

# Datasets

In [3]:
drive.mount('/content/gdrive/', force_remount=True)

Mounted at /content/gdrive/


In [4]:
df = pd.read_csv(f'{LOG_DIR}exec.txt', names=['datetime','contagem','qtd_features', '1-ds2', '1-ds3', '1-ds4', 'num_neurons', 'duracao', 'qtd_camadas','tpr2','tnr2','tpr3','tnr3','tpr4','tnr4'])

In [5]:
#df.describe()

In [6]:
df['f12'] = 1 - df['1-ds2']
df['f13'] = 1 - df['1-ds3']
df['f14'] = 1 - df['1-ds4']

In [7]:
df.drop('1-ds2', axis=1, inplace=True)
df.drop('1-ds3', axis=1, inplace=True)
df.drop('1-ds4', axis=1, inplace=True)

In [8]:
df['somaf1'] = df['f12'] + df['f13'] + df['f14']

In [9]:
df.describe()

,contagem,qtd_features,num_neurons,duracao,qtd_camadas,tpr2,tnr2,tpr3,tnr3,tpr4,tnr4,f12,f13,f14,somaf1
count,23182.000000,23182.000000,23182.000000,23182.000000,23182.000000,23182.000000,23182.000000,23182.000000,23182.000000,23182.000000,23182.000000,23182.000000,23182.000000,23182.000000,23182.000000
mean,11438.637219,24.646838,7.662928,24.692414,1.113838,0.725624,0.610386,0.841720,0.901139,0.677195,0.756006,0.679898,0.894298,0.664129,2.238326
std,6606.780588,2.763937,1.390932,11.939306,0.506997,0.136284,0.100376,0.108083,0.052025,0.142368,0.125087,0.074530,0.064269,0.122750,0.132777
min,1.000000,13.000000,2.000000,6.550000,1.000000,0.066800,0.075700,0.123800,0.483700,0.149700,0.173300,0.106400,0.216900,0.216100,1.273400
25%,5741.250000,23.000000,7.000000,15.790000,1.000000,0.653500,0.558100,0.776850,0.876100,0.560200,0.687900,0.641400,0.863500,0.579700,2.148825
50%,11467.500000,25.000000,8.000000,23.410000,1.000000,0.762700,0.603300,0.861600,0.909700,0.716750,0.795400,0.696600,0.910400,0.687500,2.257450
75%,17167.750000,26.000000,9.000000,31.140000,1.000000,0.826900,0.657600,0.927800,0.940100,0.788300,0.853100,0.736300,0.941475,0.762500,2.345000
max,22800.000000,37.000000,9.000000,176.660000,5.000000,0.967800,0.904000,0.994700,0.995400,0.972500,0.984900,0.794200,0.977300,0.883100,2.516000


In [10]:
df = df[['datetime', 'contagem', 'qtd_features', 'num_neurons', 'duracao', 'qtd_camadas', 'tpr2', 'tnr2', 'f12', 'tpr3', 'tnr3', 'f13', 'tpr4', 'tnr4', 'f14', 'somaf1']]

In [11]:
#BEST TON
df_sorted = df.sort_values(by='f12', ascending=False, inplace=False)
df_sorted.head(2)

,datetime,contagem,qtd_features,num_neurons,duracao,qtd_camadas,tpr2,tnr2,f12,tpr3,tnr3,f13,tpr4,tnr4,f14,somaf1
14214,2025-12-07 21:40:36,14030,26,8,7.21,1,0.8854,0.6557,0.7942,0.8642,0.9184,0.9140,0.3491,0.5152,0.3384,2.0466
15238,2025-12-08 03:24:09,15054,27,8,24.54,1,0.8957,0.6379,0.7934,0.8501,0.9091,0.9044,0.4462,0.4941,0.4072,2.1050


In [18]:
#BEST BOT
df_sorted = df.sort_values(by=['f13','somaf1'], ascending=False, inplace=False)
df_sorted.head(3)

,datetime,contagem,qtd_features,num_neurons,duracao,qtd_camadas,tpr2,tnr2,f12,tpr3,tnr3,f13,tpr4,tnr4,f14,somaf1
15642,2025-12-08 07:32:25,15458,22,9,31.47,1,0.4959,0.8202,0.5918,0.9912,0.8869,0.9773,0.5299,0.8120,0.5865,2.1556
20864,2025-12-10 11:21:06,20559,24,9,9.63,1,0.4986,0.7752,0.5786,0.9897,0.8916,0.9773,0.5556,0.4810,0.4789,2.0348
18338,2025-12-09 10:00:31,18111,22,9,8.80,1,0.5680,0.8064,0.6448,0.9912,0.8838,0.9769,0.5220,0.7664,0.5594,2.1811


In [13]:
#BEST CSE
df_sorted = df.sort_values(by='f14', ascending=False, inplace=False)
df_sorted.head(2)

,datetime,contagem,qtd_features,num_neurons,duracao,qtd_camadas,tpr2,tnr2,f12,tpr3,tnr3,f13,tpr4,tnr4,f14,somaf1
21237,2025-12-10 12:58:10,20932,21,9,15.92,1,0.6543,0.5772,0.6300,0.8421,0.8720,0.8939,0.8384,0.9590,0.8831,2.4070
19318,2025-12-09 15:51:29,19091,23,9,25.12,1,0.6017,0.5778,0.5946,0.8879,0.7992,0.9089,0.8424,0.9546,0.8824,2.3859


In [14]:
#BEST Soma
df_sorted = df.sort_values(by='somaf1', ascending=False, inplace=False)
df_sorted.head(2)

,datetime,contagem,qtd_features,num_neurons,duracao,qtd_camadas,tpr2,tnr2,f12,tpr3,tnr3,f13,tpr4,tnr4,f14,somaf1
12293,2025-12-07 02:12:10,12123,26,9,32.85,1,0.8226,0.5196,0.7144,0.9505,0.8745,0.9544,0.8561,0.8881,0.8472,2.5160
7010,2025-12-05 04:24:01,6911,20,6,46.41,1,0.9206,0.4700,0.7513,0.9123,0.9040,0.9387,0.8439,0.8415,0.8124,2.5024


In [15]:
#Filtra para tentar encontrar
#df_sorted = df.sort_values(by='f12', ascending=False, inplace=False)
#df_filtro = df_sorted[df_sorted['f12']>=0.71]
#df_filtro